# Fine-Tuning du Modèle SpeechCoach (Qwen-2.5 1.5B)

Ce notebook implémente le pipeline complet pour l'ajustement fin (fine-tuning) du modèle Qwen-2.5 sur le jeu de données SpeechCoach.

**Objectifs :**
- Préparation du jeu de données au format ChatML.
- Entraînement optimisé via la méthode QLoRA.
- Suivi des métriques de performance et de convergence.
- Fusion des poids et export au format GGUF pour exploitation locale.

## 1. Installation des bibliothèques nécessaires

In [ ]:
# Note: On fige les versions pour garantir la compatibilité totale avec Google Colab
!pip install -q -U transformers peft trl accelerate bitsandbytes datasets sentencepiece
!pip install -q -U scikit-learn matplotlib "pandas==2.2.2"

## 2. Configuration de l'environnement et accès aux données

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
print("Chargement du fichier coach_dataset_v2.jsonl :")
uploaded = files.upload()

## 3. Prétraitement du jeu de données

In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_to_chatml(example):
    """Conversion des entrées au format de dialogue ChatML."""
    system_prompt = (
        "Tu es un coach de prise de parole. Réponds uniquement en JSON valide avec les clés : bilan_global, point_prioritaire, encouragement."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example['input']},
        {"role": "assistant", "content": json.dumps(example['target'], ensure_ascii=False)}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

# Chargement du jeu de données
dataset = load_dataset("json", data_files="coach_dataset_v2.jsonl", split="train")

# Division en ensembles d'entraînement et de validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Application du template de dialogue
dataset = dataset.map(format_to_chatml)

print(f"Taille de l'ensemble d'entraînement : {len(dataset['train'])}")
print(f"Taille de l'ensemble de test : {len(dataset['test'])}")

## 4. Configuration de la méthode QLoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Phase d'entraînement

In [ ]:
from trl import SFTTrainer, SFTConfig

# Configuration de l'entraînement via SFTConfig
training_params = SFTConfig(
    output_dir="./speechcoach-qwen-qlora",
    dataset_text_field="text",
    max_length=1024,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    max_steps=100,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    packing=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_params
)

trainer.train()

## 6. Analyse des courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(trainer.state.log_history)

plt.figure(figsize=(10, 5))
train_rows = history[history['loss'].notna()]
eval_rows = history[history['eval_loss'].notna()]

plt.plot(train_rows['step'], train_rows['loss'], label='Training Loss', marker='o')
plt.plot(eval_rows['step'], eval_rows['eval_loss'], label='Validation Loss', marker='x')

plt.title("Évolution des fonctions de perte (Loss)")
plt.xlabel("Étapes (Steps)")
plt.ylabel("Perte (Loss)")
plt.legend()
plt.grid(True)
plt.show()

## 7. Fusion des poids et export du modèle

In [ ]:
import gc
import shutil
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Sauvegarde des adaptateurs LoRA
adapter_dir = "final_adapter"
trainer.model.save_pretrained(adapter_dir)

# Libération de la mémoire vidéo (VRAM)
del model, trainer
gc.collect()
torch.cuda.empty_cache()

# Chargement du modèle de base en haute précision pour la fusion
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={ "": "cpu" },
)
model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = model.merge_and_unload()

# Sauvegarde du modèle fusionné au format HuggingFace
hf_merged_path = "speechcoach-hf-merged"
merged_model.save_pretrained(hf_merged_path)
tokenizer.save_pretrained(hf_merged_path)

print("Modèle fusionné avec succès.")

## 8. Exportation au format GGUF (Optimisation pour Ollama)

In [ ]:
# Compilation de llama.cpp pour la conversion
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && make clean && make -j

# Conversion du modèle fusionné en format GGUF (FP16)
!python llama.cpp/convert_hf_to_gguf.py speechcoach-hf-merged --outfile speechcoach-f16.gguf --outtype f16

# Quantification du modèle pour réduire le poids (Q4_K_M)
final_path = "/content/drive/MyDrive/SpeechCoach_Qwen2.5_1.5B_Q4KM.gguf"
!./llama.cpp/llama-quantize speechcoach-f16.gguf {final_path} q4_k_m

print(f"Le fichier GGUF a été sauvegardé sur Google Drive : {final_path}")